<a href="https://colab.research.google.com/github/J4SIB/ai-course-gp/blob/main/Kopia_notatnika_Untitled25.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
!pip install transformers datasets

In [2]:
from transformers import AutoTokenizer, GPT2LMHeadModel, pipeline, Trainer, TrainingArguments
import pandas as pd
from datasets import Dataset

model_name = "flax-community/papuGaPT2"

# Tokenizer wspólny dla obu modeli
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# Model przed treningiem
model_orig = GPT2LMHeadModel.from_pretrained(model_name)
model_orig.resize_token_embeddings(len(tokenizer))

# Model, który będziemy trenować
model_tuned = GPT2LMHeadModel.from_pretrained(model_name)
model_tuned.resize_token_embeddings(len(tokenizer))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/864 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/888k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/547k [00:00<?, ?B/s]

...

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: flax-community/papuGaPT2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.bias        | UNEXPECTED |  | 
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/510M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: flax-community/papuGaPT2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.bias        | UNEXPECTED |  | 
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding(50257, 768)

In [3]:

generator_orig = pipeline("text-generation", model=model_orig, tokenizer=tokenizer)

prompt = "Gra:Terraria\nOpis:"

result = generator_orig(
    prompt,
    max_new_tokens=80,
    temperature=0.8
)

print("Odpowiedź PRZED treningiem:")
print(result[0]["generated_text"])

Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=80) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Odpowiedź PRZED treningiem:
Gra:Terraria
Opis:Terraria to gra karciana, w której pokierujemy własnym losem. Przechodzimy wszystkie poziomy i przenosimy się do kolejnych lokacji. Każda z nich będzie wyjątkowa i pomoże nam rozwijać swój biznes, o ile zdecydujemy się na to samo co my. Za zarobione pieniądze, budujemy nowe miejsca pracy, a także kupujemy nowe umiejętności i nowe przedmioty. Po wybraniu odpowiedniej lokacji, możemy też spróbować swoich


In [4]:
with open("gry.txt", "r", encoding="utf-8") as file:
  file_content = file.read()

# print(file_content)

texts = [block.strip() for block in file_content.split("===") if block.strip()]
print(texts)

df = pd.DataFrame({"text": texts})
dataset = Dataset.from_pandas(df)
print(dataset)

tokenizer.pad_token = tokenizer.eos_token

def tokenize(batch):
  tokens = tokenizer(batch["text"], padding=True, truncation=True, max_length=512)
  tokens["labels"] = tokens["input_ids"].copy()
  return tokens

tokenized_dataset = dataset.map(tokenize, batched=True)
print(tokenized_dataset)

["Gra: Baldur's Gate 3\nOpis: Baldur's Gate 3 to epicka gra RPG osadzona w uniwersum Dungeons & Dragons. Gracz wciela się w bohatera z tajemniczym pasożytem w głowie, który daje nadludzkie moce. Gra oferuje ogromną swobodę wyborów, dynamiczne walki turowe i rozbudowaną fabułę z wieloma zakończeniami.", 'Gra: Hogwarts Legacy\nOpis: Hogwarts Legacy przenosi gracza do XIX-wiecznej szkoły magii w świecie Harry’ego Pottera. Jako uczeń tworzysz własnego bohatera, uczysz się zaklęć, odkrywasz tajemnice i walczysz z czarną magią. Gra zachwyca szczegółowym światem i możliwością eksploracji poza murami Hogwartu.', 'Gra: Starfield\nOpis: Starfield to pierwsza nowa marka RPG od Bethesdy od ponad 25 lat, osadzona w kosmosie. Gracz odkrywa galaktykę, odwiedza setki planet i tworzy własne statki kosmiczne. Tytuł oferuje rozbudowany system craftingu, frakcji i nieliniową narrację.', 'Gra: The Legend of Zelda: Tears of the Kingdom\nOpis: Tears of the Kingdom to kontynuacja hitu Breath of the Wild, ofer

Map:   0%|          | 0/7 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 7
})


In [5]:
training_args = TrainingArguments(
    output_dir="./papuga_finetuned",
    num_train_epochs=20,
    logging_steps=10,
    save_steps=1000,
    save_total_limit=1,
    report_to="none",
    learning_rate=5e-5
)

trainer = Trainer(
    model=model_tuned,
    args=training_args,
    train_dataset=tokenized_dataset
)

trainer.train()

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
10,2.006419
20,0.530172


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=20, training_loss=1.2682955026626588, metrics={'train_runtime': 18.973, 'train_samples_per_second': 7.379, 'train_steps_per_second': 1.054, 'total_flos': 5501422080000.0, 'train_loss': 1.2682955026626588, 'epoch': 20.0})

In [8]:
generator_tuned = pipeline("text-generation", model=model_tuned, tokenizer=tokenizer)

prompt = "Gra:Minecraft\nOpis:"

result = generator_tuned(
    prompt,
    max_new_tokens=80,
    temperature=0.8
)

print("Odpowiedź PO treningiem:")
print(result[0]["generated_text"])

Both `max_new_tokens` (=80) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Odpowiedź PO treningiem:
Gra:Minecraft
Opis: Zrealizowany z rozmachem świat zachwyca swoją prostotą. Statki, mosty, dżungla, wulkany, pustynie i dżungla marzeń. Przetrwanie, wolność, braterstwo i niesamowita zabawa gwarantowane. Przetrwanie, wolność, braterstwo i niesamowita zabawa gwarantowane.
